In [1]:
import os


ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)
!unzip -o /content/tinyllama-sql-lora-adapter_r=16.zip -d {ADAPTER_DIR} # change here if you use another LoRA file

print("Directory content:")
!ls {ADAPTER_DIR}

Archive:  /content/tinyllama-sql-lora-adapter_r=16.zip
  inflating: ./tinyllama-sql-lora-adapter/chat_template.jinja  
  inflating: ./tinyllama-sql-lora-adapter/README.md  
  inflating: ./tinyllama-sql-lora-adapter/tokenizer_config.json  
  inflating: ./tinyllama-sql-lora-adapter/adapter_config.json  
  inflating: ./tinyllama-sql-lora-adapter/adapter_model.safetensors  
  inflating: ./tinyllama-sql-lora-adapter/tokenizer.json  
Directory content:
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [ ]:
# 1. Install necessary libraries
!pip install -q transformers peft accelerate gradio torchao --upgrade

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import gradio as gr
import os
import sqlite3

In [9]:

# 2. Load Model & Tokenizer
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"

print("Loading Tokenizer and Base Model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

# Check if LoRA file exists (as new Colab loses old data)
if os.path.exists(ADAPTER_DIR):
    print("LoRA Adapter found, integrating...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
else:
    print("⚠️ LoRA file not found. Running temporarily with the original Base Model.")
    model = base_model

model.eval()

# # 3. Prompt processing and SQL generation logic
# def build_prompt(schema: str, question: str) -> str:
#     system = "You are a SQL assistant. Based on the table schema and the question, reply ONLY with the SQL query, nothing else."
#     user = f"Schema:\n{schema}\n\nQuestion: {question}"
#     messages = [
#         {"role": "system", "content": system},
#         {"role": "user", "content": user},
#     ]
#     return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# @torch.no_grad()
# def generate_sql(schema, question):
#     prompt = build_prompt(schema, question)
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     output_ids = model.generate(
#         **inputs,
#         max_new_tokens=512,
#         do_sample=False,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )

#     input_length = inputs["input_ids"].shape[1]
#     new_tokens = output_ids[0][input_length:]
#     return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# # 4. Function to automatically read the DB file uploaded from Gradio
# def extract_schema(db_path):
#     if not db_path:
#         return "Please upload a database file."
#     try:
#         # db_path is the temporary file saved by Gradio upon upload
#         conn = sqlite3.connect(db_path)
#         cursor = conn.cursor()
#         cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
#         tables = cursor.fetchall()

#         schema = "\n".join([table[0] + ";" for table in tables if table[0]])
#         conn.close()

#         return schema if schema else "No tables found in the file."
#     except Exception as e:
#         return f"Error reading file: {e}"


Loading Tokenizer and Base Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LoRA Adapter found, integrating...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [10]:
# 5. Separate processing functions (Upload and Query)

def process_upload(db_file):
    """Runs immediately after the user uploads a file"""
    if not db_file:
        return "No DB file uploaded."

    db_path = db_file.name if hasattr(db_file, 'name') else db_file
    schema = extract_schema(db_path)
    return schema

@torch.no_grad()
def ask_question(schema, question):
    """Takes the user question and the extracted schema to generate SQL"""
    if not schema or "Error" in schema or "Please" in schema or "No tables" in schema:
        return "-- Please upload a valid database (.db) file first! --"
    if not question.strip():
        return "-- Please enter your question! --"

    # Keep the original prompt logic and TinyLlama model generation
    prompt = build_prompt(schema, question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]
    sql = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return sql


# 6. Build optimized UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("### Text-to-SQL AI: Direct Q&A with your Database")

    with gr.Row():
        db_upload = gr.File(label="1. Drag and drop SQLite file (.db) here", file_types=[".db", ".sqlite"])

    # Use Accordion with open=False to hide the Table Structure by default
    with gr.Accordion("View Table Structure (Auto-read)", open=False):
        schema_output = gr.Code(label="Detailed Schema", language="sql")

    with gr.Row():
        question_input = gr.Textbox(label="2. Enter your question", placeholder="Example: List the tracks that have a millisecond duration...", lines=2)

    btn = gr.Button("Generate SQL Query", variant="primary")

    with gr.Row():
        sql_output = gr.Code(label="Generated SQL Result", language="sql")

    # ================= AUTOMATIC EVENT TRIGGERS =================

    # Event 1: ON FILE UPLOAD -> Extract schema and fill the hidden block
    db_upload.upload(fn=process_upload, inputs=[db_upload], outputs=[schema_output])

    # Event 2: ON BUTTON CLICK -> Use extracted schema + question to generate SQL
    btn.click(fn=ask_question, inputs=[schema_output, question_input], outputs=[sql_output])

# Launch UI
demo.launch(share=True)

/tmp/ipykernel_1515/2668393085.py:40: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0bd65ec2c87b803f0c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
